In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

from peft import PeftModel

base_model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"
adapter_dir = "/home/dell/SLM_Engineering/Final_FP16_3000_final_adapter_smollm"

print("⏳ Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    adapter_dir,
    local_files_only=True
)

tokenizer.pad_token = tokenizer.eos_token

print("✅ Tokenizer loaded!")

print("⏳ Loading base model in FP16...")

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_dir,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("✅ Base model loaded!")

print("⏳ Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    is_trainable=False
)

model.eval()

print("✅ LoRA adapter loaded!")

print(
    f"GPU memory allocated: "
    f"{torch.cuda.memory_allocated() / (1024**2):.2f} MB"
)

print("\n" + "=" * 60)
print("✅ FINE-TUNED AutoHeal MODEL READY FOR INFERENCE")
print("=" * 60)

⏳ Loading tokenizer...
✅ Tokenizer loaded!
⏳ Loading base model in FP16...
✅ Base model loaded!
⏳ Loading LoRA adapter...
✅ LoRA adapter loaded!
GPU memory allocated: 737.21 MB

✅ FINE-TUNED AutoHeal MODEL READY FOR INFERENCE


In [2]:
import json
import re

# ============================================================
# TOOL REGISTRY — Real output messages from Tool Outputs [1]
# ============================================================
TOOL_RESPONSES = {
    # --- Diagnostic Tools (Confirmed) ---
    "GetACAdapterChargeWattageStatus": "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed",
    "CheckHeavyBatteryUsageByApps": "High battery usage was detected.",
    "UpdateDrivers": "Driver update has completed.",
    "CheckHeavyACAdapterUsageSettings": "Device is primarily used on AC adapter. Remediation is required.",
    "GetBatteryChargeStateChangesCount": "Frequent AC Power Connect/Disconnect.",
    "CheckLowEnergySettings": "Low energy settings are disabled.",

    # --- Remediation Tools (Success) ---
    "NotifyAdapterChargeWattageIsNotSufficient": "AC Adapter Charge Wattage Notification displayed successfully to user.",
    "NotifyHeavyBatteryUsageByApps": "User notified of top three apps consuming battery.",
    "UpdateBIOSBatteryProfile": "BIOS battery profile updated to primarily on AC power. Remediation successful.",
    "NotifyBatteryChargeStateChanges": "Battery Charge State Notification displayed successfully to user.",
    "SetLowEnergySettings": "All settings for low energy are optimized."
}

# --- NOT CONFIRMED responses [1] ---
NOT_CONFIRMED = {
    "GetACAdapterChargeWattageStatus": "Sufficient AC Adapter charge wattage detected. No need to display AC Adapter Charge Wattage Notification.",
    "CheckHeavyBatteryUsageByApps": "Battery usage is below the threshold.",
    "UpdateDrivers": "No drivers found for installation after driver scan.",
    "CheckHeavyACAdapterUsageSettings": "Device is not primarily used on AC adapter. No need to display Battery charge state notification.",
    "GetBatteryChargeStateChangesCount": "No frequent AC Power Connect/Disconnect detected.",
    "CheckLowEnergySettings": "Low energy settings are enabled."
}

# --- ERROR responses [1] ---
ERROR_RESPONSES = {
    "GetACAdapterChargeWattageStatus": "Adapter wattage data not found in DTP. No need to display AC Adapter Charge Wattage Notification.",
    "CheckHeavyBatteryUsageByApps": "Some error occurred while executing the script.",
    "UpdateDrivers": "Failed to scan drivers.",
    "CheckHeavyACAdapterUsageSettings": "Some error occurred while collecting AC adapter usage data. No need to display Battery charge state notification.",
    "GetBatteryChargeStateChangesCount": "Some error occurred while executing GetBatteryChargeStateChangesCount. No need to display Battery charge state notification.",
    "CheckLowEnergySettings": "Device is not on battery."
}

# --- REMEDIATION FAILURE responses [1] ---
REMEDIATION_FAILURES = {
    "NotifyAdapterChargeWattageIsNotSufficient": "Failed to display AC Adapter Charge Wattage Notification.",
    "NotifyHeavyBatteryUsageByApps": "Failed to notify user of top three apps consuming battery.",
    "UpdateBIOSBatteryProfile": "Couldn't complete the BIOS battery profile update due to a script error.",
    "NotifyBatteryChargeStateChanges": "Failed to display Battery Charge State Notification.",
    "SetLowEnergySettings": "Could not display low energy settings notification."
}

def execute_tool(tool_name):
    """Execute tool — returns JSON-wrapped result matching training format"""
    if tool_name in TOOL_RESPONSES:
        return json.dumps({"result": TOOL_RESPONSES[tool_name]})
    return json.dumps({"result": f"Unknown tool: {tool_name}"})

def set_not_confirmed(tool_name):
    """Set diagnostic to return NOT confirmed"""
    if tool_name in NOT_CONFIRMED:
        TOOL_RESPONSES[tool_name] = NOT_CONFIRMED[tool_name]
        print(f"  [CONFIG] {tool_name} -> NOT CONFIRMED")

def set_error(tool_name):
    """Set diagnostic to return ERROR"""
    if tool_name in ERROR_RESPONSES:
        TOOL_RESPONSES[tool_name] = ERROR_RESPONSES[tool_name]
        print(f"  [CONFIG] {tool_name} -> ERROR")

def set_remediation_failure(tool_name):
    """Set remediation to return FAILURE"""
    if tool_name in REMEDIATION_FAILURES:
        TOOL_RESPONSES[tool_name] = REMEDIATION_FAILURES[tool_name]
        print(f"  [CONFIG] {tool_name} -> FAILURE")

def reset_all():
    """Reset all tools to confirmed/success"""
    global TOOL_RESPONSES
    TOOL_RESPONSES = {
        "GetACAdapterChargeWattageStatus": "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed",
        "CheckHeavyBatteryUsageByApps": "High battery usage was detected.",
        "UpdateDrivers": "Driver update has completed.",
        "CheckHeavyACAdapterUsageSettings": "Device is primarily used on AC adapter. Remediation is required.",
        "GetBatteryChargeStateChangesCount": "Frequent AC Power Connect/Disconnect.",
        "CheckLowEnergySettings": "Low energy settings are disabled.",
        "NotifyAdapterChargeWattageIsNotSufficient": "AC Adapter Charge Wattage Notification displayed successfully to user.",
        "NotifyHeavyBatteryUsageByApps": "User notified of top three apps consuming battery.",
        "UpdateBIOSBatteryProfile": "BIOS battery profile updated to primarily on AC power. Remediation successful.",
        "NotifyBatteryChargeStateChanges": "Battery Charge State Notification displayed successfully to user.",
        "SetLowEnergySettings": "All settings for low energy are optimized."
    }
    print("  [CONFIG] All tools reset to CONFIRMED/SUCCESS")

print("Cell 1: Tool Registry loaded successfully")

Cell 1: Tool Registry loaded successfully


In [3]:
import time
import torch

# ============================================================
# SYSTEM PROMPT — Matches your SmolLM training format
# ============================================================
SYSTEM_PROMPT = """You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.

CORE RULES
- REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.
- TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.
- AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.

REASONING PROCESS
1. Hypothesis & Diagnostics:
   Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.
2. Evidence Evaluation & Action:
   Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.
3. Resolution:
   Confirm the remediation outcome and output your final assessment.

TOOL FORMAT: {"tool": "<tool_name>"}

FINAL ANSWER FORMAT:
Root Cause: <Explanation of the verified issue>
Evidence:
- <DiagnosticToolName> returned: "<Diagnostic output>"
Remediation:
- Function: <RemediationToolName>
- Execution Result: <Remediation output>
- Status: EXECUTED"""
# ============================================================
# TOOL CALL DETECTOR — Matches {"tool": "<name>"} format
# ============================================================
TOOL_PATTERN = r'\{"tool":\s*"([^"]+)"\}'

def detect_tool_call(text):
    """Detect tool call in model output"""
    matches = list(re.finditer(TOOL_PATTERN, text))
    if matches:
        match = matches[-1]
        return {"tool_name": match.group(1), "end_pos": match.end()}
    return None

def trim_at_tool_call(text):
    """Trim text at tool call boundary"""
    match = re.search(TOOL_PATTERN, text)
    if match:
        return text[:match.end()].strip(), True
    return text.strip(), False

# ============================================================
# USER PROMPT BUILDER
# ============================================================
def to_user_prompt(data):
    """Build user prompt matching training format"""
    parts = []
    if "calibrated_confidence" in data:
        parts.append(f"calibrated_confidence: {data['calibrated_confidence']}")
    if "deviation_score" in data:
        parts.append(f"deviation_score: {data['deviation_score']}")
    if "evidence_tier" in data:
        parts.append(f"evidence_tier: {data['evidence_tier']}")
    parts.append("")
    parts.append("shap_features:")
    for f in data["shap_features"]:
        parts.append(f"- {f['feature']}: {f['shap_value']}")
    return "\n".join(parts)

# ============================================================
# MULTI-TURN INFERENCE LOOP
# Key: Tool responses come as role="user" with [TOOL_RESPONSE] prefix
# ============================================================
def run_test(test_input, model, tokenizer, max_tool_calls=5):
    """Run inference matching SmolLM training format"""
    user_prompt = to_user_prompt(test_input)

    print("=" * 60)
    print(f"  TEST: {test_input.get('name', 'Unnamed')}")
    print("=" * 60)
    print(f"  INPUT:\n{user_prompt}")
    print("=" * 60)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    tool_calls_made = 0
    start = time.time()

    while tool_calls_made < max_tool_calls:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=True,
                temperature=0.1,
                top_p=0.9,
                repetition_penalty=1.1
            )

        new_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ).strip()

        trimmed_text, has_tool_call = trim_at_tool_call(new_text)

        if has_tool_call:
            tool_call = detect_tool_call(trimmed_text)
            if tool_call:
                tool_calls_made += 1

                # Add assistant turn (ends at ACTION)
                messages.append({"role": "assistant", "content": trimmed_text})

                # Execute tool from registry
                result = execute_tool(tool_call["tool_name"])

                # Add tool response as USER with [TOOL_RESPONSE] prefix
                # This matches your SmolLM training format
                messages.append({
                    "role": "user",
                    "content": f"[TOOL_RESPONSE] {result}"
                })
            else:
                messages.append({"role": "assistant", "content": new_text})
                break
        else:
            messages.append({"role": "assistant", "content": new_text})
            break

    elapsed = time.time() - start

    # Print conversation in chronological order
    print(f"\n{'='*60}")
    print("  CONVERSATION")
    print("=" * 60)
    turn = 0
    for msg in messages:
        if msg["role"] == "system":
            continue
        elif msg["role"] == "user" and not msg["content"].startswith("[TOOL_RESPONSE]"):
            continue
        elif msg["role"] == "assistant":
            turn += 1
            print(f"\n[ASSISTANT TURN {turn}]")
            print(msg["content"])
            print("-" * 40)
        elif msg["role"] == "user" and msg["content"].startswith("[TOOL_RESPONSE]"):
            print(f"[TOOL RESULT] {msg['content']}")
            print("-" * 40)

    print("=" * 60)
    print(f"Time: {elapsed:.2f}s | Tool calls: {tool_calls_made}")
    print("=" * 60)

    return messages

print("Cell 2: Inference engine loaded successfully")

Cell 2: Inference engine loaded successfully


In [4]:
# ============================================================
# SCENARIO 1 TEST CASES — Confirmed for each diagnostic [1][2]
# ============================================================

# Test 1: GetACAdapterChargeWattageStatus — Confirmed
test_ac_adapter = {
    "name": "AC Adapter Wattage — Confirmed",
    # "calibrated_confidence": 0.92,
    # "deviation_score": 0.93,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "AttachedAdapterWattage", "shap_value": 0.54},
        {"feature": "AppName", "shap_value": 0.25},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.21}
    ]
}

# Test 2: CheckHeavyBatteryUsageByApps — Confirmed
test_heavy_apps = {
    "name": "Heavy Battery Usage by Apps — Confirmed",
    # "calibrated_confidence": 0.88,
    # "deviation_score": 0.89,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "CpuEnergy", "shap_value": 0.48},
        {"feature": "EppValue", "shap_value": 0.30},
        {"feature": "EppValue", "shap_value": 0.22}
    ]
}

# Test 3: UpdateDrivers — Confirmed
test_outdated_driver = {
    "name": "Outdated Driver — Confirmed",
    # "calibrated_confidence": 0.85,
    # "deviation_score": 0.83,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "bios_full_version", "shap_value": 0.45},
        {"feature": "release_date", "shap_value": 0.32},
        {"feature": "bios_microcode_revision", "shap_value": 0.23}
    ]
}

# Test 4: CheckHeavyACAdapterUsageSettings — Confirmed
test_heavy_ac = {
    "name": "Heavy AC Adapter Usage — Confirmed",
    # "calibrated_confidence": 0.84,
    # "deviation_score": 0.82,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "bios_setting_primarily_ac", "shap_value": 0.50},
        {"feature": "ac_connect_disconnect_events_24h", "shap_value": 0.28},
        {"feature": "EppValue", "shap_value": 0.22}
    ]
}


# Test 6: CheckLowEnergySettings — Confirmed
test_energy_settings = {
    "name": "Energy Settings Not Configured — Confirmed",
    # "calibrated_confidence": 0.92,
    # "deviation_score": 0.93,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "EppValue", "shap_value": 0.54},
        {"feature": "CpuEnergy", "shap_value": 0.25},
        {"feature": "AppName", "shap_value": 0.21}
    ]
}



test_heavy_apps2 = {
    "name": "Heavy Battery Usage by Apps — Confirmed",
    # "calibrated_confidence": 0.88,
    # "deviation_score": 0.89,
    # "evidence_tier": "STRONG",
    "shap_features": [
        {"feature": "CpuEnergy", "shap_value": 0.88},
        {"feature": "AttachedAdapterWattage", "shap_value": 0.30},
        {"feature": "EppValue", "shap_value": 0.22}
    ]
}

print("Cell 3: Test cases loaded successfully")
print(f"  Available tests: {6}")
print("  Run: messages = run_test(test_ac_adapter, model, tokenizer)")

Cell 3: Test cases loaded successfully
  Available tests: 6
  Run: messages = run_test(test_ac_adapter, model, tokenizer)


In [5]:
run_test(test_heavy_apps, model, tokenizer)

  TEST: Heavy Battery Usage by Apps — Confirmed
  INPUT:

shap_features:
- CpuEnergy: 0.48
- EppValue: 0.3
- EppValue: 0.22

  CONVERSATION

[ASSISTANT TURN 1]
THOUGHT 1:
The highest SHAP feature is CpuEnergy.

Looking up CpuEnergy
→ It is EXCLUSIVE
→ It maps to the diagnostic tool: CheckHeavyBatteryUsageByApps

I will call the diagnostic tool: CheckHeavyBatteryUsageByApps.

ACTION 1:
{"tool":"CheckHeavyBatteryUsageByApps"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "High battery usage was detected."}
----------------------------------------

[ASSISTANT TURN 2]
THOUGHT 2:
Previous tool CheckHeavyBatteryUsageByApps returned "High battery usage was detected." (evaluation: POSITIVE).

It maps to the remediation tool: NotifyHeavyBatteryUsageByApps

I will call the remediation tool: NotifyHeavyBatteryUsageByApps.

ACTION 2:
{"tool":"NotifyHeavyBatteryUsageByApps"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "User n

[{'role': 'system',
  'content': 'You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.\n\nCORE RULES\n- REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.\n- TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.\n- AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.\n\nREASONING PROCESS\n1. Hypothesis & Diagnostics:\n   Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.\n2. Evidence Evaluation & Action:\n   Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.\n3. Resolution

In [6]:
run_test(test_heavy_apps2, model, tokenizer)

  TEST: Heavy Battery Usage by Apps — Confirmed
  INPUT:

shap_features:
- CpuEnergy: 0.88
- AttachedAdapterWattage: 0.3
- EppValue: 0.22

  CONVERSATION

[ASSISTANT TURN 1]
THOUGHT 1:
The highest SHAP feature is CpuEnergy.

Looking up CpuEnergy
→ It is EXCLUSIVE
→ It maps to the diagnostic tool: CheckHeavyBatteryUsageByApps

I will call the diagnostic tool: CheckHeavyBatteryUsageByApps.

ACTION 1:
{"tool":"CheckHeavyBatteryUsageByApps"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "High battery usage was detected."}
----------------------------------------

[ASSISTANT TURN 2]
THOUGHT 2:
Previous tool CheckHeavyBatteryUsageByApps returned "High battery usage was detected." (evaluation: POSITIVE).

It maps to the remediation tool: NotifyHeavyBatteryUsageByApps

I will call the remediation tool: NotifyHeavyBatteryUsageByApps.

ACTION 2:
{"tool":"NotifyHeavyBatteryUsageByApps"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"re

[{'role': 'system',
  'content': 'You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.\n\nCORE RULES\n- REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.\n- TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.\n- AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.\n\nREASONING PROCESS\n1. Hypothesis & Diagnostics:\n   Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.\n2. Evidence Evaluation & Action:\n   Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.\n3. Resolution

In [7]:
run_test(test_ac_adapter, model, tokenizer)

  TEST: AC Adapter Wattage — Confirmed
  INPUT:

shap_features:
- AttachedAdapterWattage: 0.54
- AppName: 0.25
- ac_connect_disconnect_events_24h: 0.21

  CONVERSATION

[ASSISTANT TURN 1]
THOUGHT 1:
The highest SHAP feature is AttachedAdapterWattage.

Looking up AttachedAdapterWattage
→ It is EXCLUSIVE
→ It maps to the diagnostic tool: GetACAdapterChargeWattageStatus

I will call the diagnostic tool: GetACAdapterChargeWattageStatus.

ACTION 1:
{"tool":"GetACAdapterChargeWattageStatus"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed"}
----------------------------------------

[ASSISTANT TURN 2]
THOUGHT 2:
Previous tool GetACAdapterChargeWattageStatus returned "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed" (evaluation: POSITIVE).

It maps to the remediation tool: NotifyAdapterChargeWa

[{'role': 'system',
  'content': 'You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.\n\nCORE RULES\n- REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.\n- TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.\n- AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.\n\nREASONING PROCESS\n1. Hypothesis & Diagnostics:\n   Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.\n2. Evidence Evaluation & Action:\n   Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.\n3. Resolution

In [8]:
run_test(test_heavy_ac, model, tokenizer)


  TEST: Heavy AC Adapter Usage — Confirmed
  INPUT:

shap_features:
- bios_setting_primarily_ac: 0.5
- ac_connect_disconnect_events_24h: 0.28
- EppValue: 0.22

  CONVERSATION

[ASSISTANT TURN 1]
THOUGHT 1:
The highest SHAP feature is bios_setting_primarily_ac.

Looking up bios_setting_primarily_ac
→ It is EXCLUSIVE
→ It maps to the diagnostic tool: CheckHeavyACAdapterUsageSettings

I will call the diagnostic tool: CheckHeavyACAdapterUsageSettings.

ACTION 1:
{"tool":"CheckHeavyACAdapterUsageSettings"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "Device is primarily used on AC adapter. Remediation is required."}
----------------------------------------

[ASSISTANT TURN 2]
THOUGHT 2:
Previous tool CheckHeavyACAdapterUsageSettings returned "Device is primarily used on AC adapter. Remediation is required." (evaluation: POSITIVE).

It maps to the remediation tool: UpdateBIOSBatteryProfile

I will call the remediation tool: UpdateBIOSBatteryProfile.

ACT

[{'role': 'system',
  'content': 'You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.\n\nCORE RULES\n- REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.\n- TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.\n- AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.\n\nREASONING PROCESS\n1. Hypothesis & Diagnostics:\n   Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.\n2. Evidence Evaluation & Action:\n   Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.\n3. Resolution

In [9]:
run_test(test_energy_settings, model, tokenizer)

  TEST: Energy Settings Not Configured — Confirmed
  INPUT:

shap_features:
- EppValue: 0.54
- CpuEnergy: 0.25
- AppName: 0.21

  CONVERSATION

[ASSISTANT TURN 1]
THOUGHT 1:
The highest SHAP feature is EppValue.

Looking up EppValue
→ It is EXCLUSIVE
→ It maps to the diagnostic tool: CheckLowEnergySettings

I will call the diagnostic tool: CheckLowEnergySettings.

ACTION 1:
{"tool":"CheckLowEnergySettings"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "Low energy settings are disabled."}
----------------------------------------

[ASSISTANT TURN 2]
THOUGHT 2:
Previous tool CheckLowEnergySettings returned "Low energy settings are disabled." (evaluation: POSITIVE).

It maps to the remediation tool: SetLowEnergySettings

I will call the remediation tool: SetLowEnergySettings.

ACTION 2:
{"tool":"SetLowEnergySettings"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "All settings for low energy are optimized."}
---------

[{'role': 'system',
  'content': 'You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.\n\nCORE RULES\n- REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION.\n- TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run.\n- AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.\n\nREASONING PROCESS\n1. Hypothesis & Diagnostics:\n   Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.\n2. Evidence Evaluation & Action:\n   Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool.\n3. Resolution

In [10]:
# ============================================================
# RUN ALL SCENARIO 1 TESTS — Each diagnostic confirmed [1][2]
# ============================================================

all_tests = [
    test_ac_adapter,
    test_heavy_apps,
    test_outdated_driver,
    test_heavy_ac,
    test_energy_settings
]

results = {}

for test in all_tests:
    reset_all()
    print(f"\n>>> Running: {test['name']}")
    messages = run_test(test, model, tokenizer)
    results[test['name']] = messages
    print("\n")

# ============================================================
# SUMMARY
# ============================================================
print("=" * 60)
print("  TEST SUMMARY")
print("=" * 60)
for name, msgs in results.items():
    tool_count = sum(1 for m in msgs
                     if m["role"] == "user"
                     and m["content"].startswith("[TOOL_RESPONSE]"))
    has_final = any("FINAL ANSWER" in m["content"]
                    for m in msgs if m["role"] == "assistant")
    status = "PASS" if has_final and tool_count >= 2 else "CHECK"
    print(f"  {status} | {name} | Tool calls: {tool_count}")
print("=" * 60)

  [CONFIG] All tools reset to CONFIRMED/SUCCESS

>>> Running: AC Adapter Wattage — Confirmed
  TEST: AC Adapter Wattage — Confirmed
  INPUT:

shap_features:
- AttachedAdapterWattage: 0.54
- AppName: 0.25
- ac_connect_disconnect_events_24h: 0.21

  CONVERSATION

[ASSISTANT TURN 1]
THOUGHT 1:
The highest SHAP feature is AttachedAdapterWattage.

Looking up AttachedAdapterWattage
→ It is EXCLUSIVE
→ It maps to the diagnostic tool: GetACAdapterChargeWattageStatus

I will call the diagnostic tool: GetACAdapterChargeWattageStatus.

ACTION 1:
{"tool":"GetACAdapterChargeWattageStatus"}
----------------------------------------
[TOOL RESULT] [TOOL_RESPONSE] {"result": "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed"}
----------------------------------------

[ASSISTANT TURN 2]
THOUGHT 2:
Previous tool GetACAdapterChargeWattageStatus returned "Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to 

In [ ]:
run